# YOLO Autoresearch Experiment Analysis

Analysis of autonomous YOLO experiments logged in `results.tsv`. The primary frontier metric is `val_map50_95`, and higher is better.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RESULTS_PATH = Path('results.tsv')
if not RESULTS_PATH.exists():
    raise FileNotFoundError('results.tsv not found in the repository root')

df = pd.read_csv(RESULTS_PATH, sep='\t')
expected_columns = [
    'commit',
    'val_map50',
    'val_map50_95',
    'precision',
    'recall',
    'memory_gb',
    'status',
    'description',
]
missing = [c for c in expected_columns if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

numeric_cols = ['val_map50', 'val_map50_95', 'precision', 'recall', 'memory_gb']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['status'] = df['status'].astype(str).str.strip().str.upper()
df['description'] = df['description'].fillna('').astype(str).str.strip()

print(f'Total experiments: {len(df)}')
print(f'Columns: {list(df.columns)}')
if df.empty:
    print('results.tsv only contains the header so far.')
else:
    print(df.head().to_string(index=False))


In [ ]:
counts = df['status'].value_counts()
print('Experiment outcomes:')
if counts.empty:
    print('No experiment outcomes yet.')
else:
    print(counts.to_string())

n_keep = counts.get('KEEP', 0)
n_discard = counts.get('DISCARD', 0)
n_crash = counts.get('CRASH', 0)
n_decided = n_keep + n_discard
if n_decided:
    print(f'\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}')
if len(df):
    print(f'Crash rate: {n_crash}/{len(df)} = {n_crash / len(df):.1%}')


In [ ]:
kept = df[df['status'] == 'KEEP'].copy()
if kept.empty:
    print('No kept experiments yet.')
else:
    display_cols = ['commit', 'val_map50', 'val_map50_95', 'precision', 'recall', 'memory_gb', 'description']
    print(kept[display_cols].reset_index(drop=True).to_string(index=False))


## Validation mAP50-95 Frontier

The running maximum of `val_map50_95` shows the best experiment found so far. Kept runs should define the frontier.


In [ ]:
valid = df[df['status'] != 'CRASH'].copy().reset_index(drop=True)
if valid.empty:
    print('No non-crash experiments to plot yet.')
else:
    baseline = valid.loc[0, 'val_map50_95']
    valid['running_best'] = valid['val_map50_95'].cummax()
    kept_mask = valid['status'] == 'KEEP'
    discard_mask = valid['status'] == 'DISCARD'

    fig, ax = plt.subplots(figsize=(16, 8))
    ax.scatter(valid.index[discard_mask], valid.loc[discard_mask, 'val_map50_95'], c='#cccccc', s=18, alpha=0.6, label='Discarded', zorder=2)
    ax.scatter(valid.index[kept_mask], valid.loc[kept_mask, 'val_map50_95'], c='#2ecc71', s=55, edgecolors='black', linewidths=0.5, label='Kept', zorder=4)
    ax.step(valid.index, valid['running_best'], where='post', color='#1f7a3a', linewidth=2.0, alpha=0.85, label='Running best', zorder=3)

    for idx, row in valid[kept_mask].iterrows():
        desc = row['description'] or row['commit']
        if len(desc) > 45:
            desc = desc[:42] + '...'
        ax.annotate(desc, (idx, row['val_map50_95']), textcoords='offset points', xytext=(6, 6), fontsize=8, color='#1f7a3a', rotation=25, ha='left', va='bottom')

    best = valid['running_best'].max()
    margin = max((best - baseline) * 0.2, 0.005)
    ax.set_ylim(max(0.0, min(valid['val_map50_95']) - margin), best + margin)
    ax.set_xlabel('Experiment #', fontsize=12)
    ax.set_ylabel('Validation mAP50-95 (higher is better)', fontsize=12)
    ax.set_title(f'YOLO Autoresearch Progress: baseline={baseline:.4f}, best={best:.4f}', fontsize=14)
    ax.grid(True, alpha=0.2)
    ax.legend(loc='lower right', fontsize=9)
    plt.tight_layout()
    plt.savefig('progress.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved plot to progress.png')


## Summary Statistics

These numbers compare the baseline against the current best kept experiment.


In [ ]:
kept = df[df['status'] == 'KEEP'].copy()
if kept.empty:
    print('No kept experiments available yet.')
else:
    baseline_row = kept.iloc[0]
    best_row = kept.loc[kept['val_map50_95'].idxmax()]
    delta = best_row['val_map50_95'] - baseline_row['val_map50_95']
    rel = delta / baseline_row['val_map50_95'] if baseline_row['val_map50_95'] else np.nan

    print(f"Baseline val_map50_95: {baseline_row['val_map50_95']:.6f}")
    print(f"Best val_map50_95:     {best_row['val_map50_95']:.6f}")
    print(f"Absolute gain:        {delta:+.6f}")
    print(f"Relative gain:        {rel:+.2%}")
    print(f"Best experiment:      {best_row['description']}")
    print(f"Best commit:          {best_row['commit']}")


## Top Hits

Each kept experiment is compared with the previous kept experiment to show which changes moved the frontier the most.


In [ ]:
kept = df[df['status'] == 'KEEP'].copy().reset_index(drop=True)
if len(kept) <= 1:
    print('Need at least two kept experiments to rank improvements.')
else:
    hits = kept.copy()
    hits['prev_map50_95'] = hits['val_map50_95'].shift(1)
    hits['delta'] = hits['val_map50_95'] - hits['prev_map50_95']
    hits = hits.iloc[1:].sort_values('delta', ascending=False)
    print(f"{'Rank':>4}  {'Delta':>9}  {'mAP50-95':>10}  Description")
    print('-' * 90)
    for rank, (_, row) in enumerate(hits.iterrows(), 1):
        print(f"{rank:4d}  {row['delta']:+.6f}  {row['val_map50_95']:.6f}  {row['description']}")

    print(f"\nTotal frontier gain: {hits['delta'].sum():+.6f}")
